In [ ]:
try:
    import langchain
    import dotenv
    import openai
    print(" 所有依赖包安装成功")
    print(f"LangChain 版本: {langchain.__version__}")
except ImportError as e:
    print(f" 依赖包导入失败: {e}")

import sys
print(f"Python 版本: {sys.version}")
print(f"版本信息: {sys.version_info}")

# 检查是否满足最低版本要求
if sys.version_info >= (3, 10):
    print(" Python 版本满足要求（>= 3.10）")
else:
    print("Python 版本过低，请升级至 3.10 或更高版本")

from dotenv import load_dotenv
import os

# 加载 .env 文件中的环境变量
load_dotenv()

# 读取 API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

# 检查 API Key 是否成功加载
if api_key:
    print("API Key 加载成功")
    print(f"API Key: {api_key[:4]}...{api_key[-4:]}")  # 仅显示前4和后4字符，保护隐私
else:
    print("API Key 加载失败，请检查 .env 文件是否正确配置")

from pathlib import Path
from base_agent import scan_skills, initialize_agent, chat, chat_stream

# 指定 Skills 目录路径
skills_dir = Path("./Skills")

# 扫描Skills目录，获取所有技能
skills_snapshot = scan_skills(skills_dir)

print("扫描到的技能列表:")
print(skills_snapshot)


 所有依赖包安装成功
LangChain 版本: 1.2.15
Python 版本: 3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:37) [MSC v.1929 64 bit (AMD64)]
版本信息: sys.version_info(major=3, minor=13, micro=7, releaselevel='final', serial=0)
 Python 版本满足要求（>= 3.10）
API Key 加载成功
API Key: sk-3...32c2


d:\anaconda3\envs\pytorch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📋 Skills snapshot: 10 skills found
扫描到的技能列表:
<available_skills>
  <skill>
    <name>alloy-design-orchestrator</name>
    <description>Coordinate the full high-temperature alloy intelligent-design workflow across specialized skills. Use when user needs to orchestrate Excel intake, Thermo-Calc thermal property calculation, ML model selection, SHAP feature screening, NSGA-III multi-objective optimization, and optional UTS/EL mechanical-property filtering while keeping user confirmations explicit.</description>
    <location>Skills\alloy-design-orchestrator\SKILL.md</location>
  </skill>
  <skill>
    <name>alloy-excel-intake</name>
    <description>Inspect high-temperature alloy Excel workbooks before Thermo-Calc or ML work. Read uploaded .xlsx/.xls/.csv alloy datasets, decide whether rows contain only alloy composition columns, identify extra metadata or label columns, ask the user before dropping columns, and create a composition-only working file without overwriting the source.</descri

### 初始化Agent实例

In [3]:
# 初始化 Agent

agent = initialize_agent(
    api_key = api_key,       # API key (从环境变量加载)
    base_url = "https://api.deepseek.com", 
    model = "deepseek-chat",
    temperature = 0.0,
    skills_dir = skills_dir,
    base_dir = Path(".")
)

📋 Skills snapshot: 10 skills found
🤖 Agent initialized with 3 tools (model: deepseek-chat)


# 多轮对话测试


In [ ]:
async def multi_turn_chat():
    print("多轮对话已启动，输入 exit / quit / 退出 可结束。\n")

    history = []

    while True:
        user_input = input("\n你: ").strip()

        if user_input.lower() in ["exit", "quit", "q", "退出", "结束"]:
            print("对话结束")
            break

        if not user_input:
            continue

        prompt = build_prompt(history, user_input)

        print("Agent:", end=" ", flush=True)

        assistant_reply = ""

        async for event in chat_stream(agent, prompt):
            if event["type"] == "token":
                content = event["content"]
                assistant_reply += content
                print(content, end="", flush=True)

        print("\n")

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": assistant_reply})


def build_prompt(history, user_input):
    prompt = "以下是用户和 Agent 的历史对话。请结合上下文继续回答。\n\n"

    for msg in history:
        if msg["role"] == "user":
            prompt += f"用户: {msg['content']}\n"
        elif msg["role"] == "assistant":
            prompt += f"Agent: {msg['content']}\n"

    prompt += f"\n用户: {user_input}\nAgent:"
    return prompt


await multi_turn_chat()

多轮对话已启动，输入 exit / quit / 退出 可结束。

Agent: 好的，我来处理这个任务。首先，我需要读取相关技能文件来了解工作流程。

让我先读取合金设计编排器（orchestrator）的技能文件，了解整体工作流程。好的，我了解了整体工作流程。现在让我读取各个子技能文件，了解具体步骤。好的，我已经了解了整个工作流程。现在让我先检查一下项目目录结构，看看有没有现有的文件。我看到已经有 `testIN718.xlsx` 文件了，而且还有 `intake_clean.xlsx` 和 `report.json`，看起来之前已经做过一些步骤。让我先检查一下这些文件的内容。好的，我现在了解了情况。让我检查一下是否有已经做过的 Thermo-Calc 计算结果。好的，目前的情况是：

1. **testIN718.xlsx** - 原始数据文件，包含 7 个元素（Cr, Fe, Nb, Al, Ti, Mo, Co），3 个合金成分样本
2. **intake_clean.xlsx** - 已经通过 `alloy-excel-intake` 处理过的纯净成分文件
3. **还没有进行 Thermo-Calc 计算** - 没有 `thermal_results.xlsx`

现在让我开始执行工作流程。首先，我需要确认用户的需求。

---

您好！我来帮您处理 **testIN718** 的铸造工艺性能热力学参数计算与优化。

根据我检查的情况，目前项目中有以下文件：
- **testIN718.xlsx** — 原始数据（3个合金成分，7个元素：Cr, Fe, Nb, Al, Ti, Mo, Co）
- **intake_clean.xlsx** — 已处理好的纯净成分文件

**整体工作流程如下：**

1. ✅ **Excel 数据检查** — 已完成（intake_clean.xlsx 已就绪）
2. ⏳ **Thermo-Calc 热力学参数计算** — 计算 4 个铸造工艺参数：
   - **Freeze Range（凝固区间）** — 越小越好
   - **Melt Viscosity（熔体粘度）** — 越小越好
   - **Surface Tension（表面张力）** — 越小越好
   - **Latent Heat（潜热）** —